# Analyze Patterns

In [1]:
# connect to Enterprise GIS
from arcgis.gis import GIS
import arcgis.geoanalytics

portal_gis = GIS("https://ndhwks6.esri.com/portal", "admin", 'esri.agp', verify_cert=False)

In [2]:
search_result1 = portal_gis.content.get('583ef29330e74212862fa71104d39ca8')
search_result1

<Item title:"calgary_no_southland_solar" type:Feature Layer Collection owner:admin>

In [3]:
data_layer = search_result1.layers[0]

In [4]:
search_result2 = portal_gis.content.search("bigDataFileShares_ServiceCallsOrleans", 
                                          item_type = "big data file share", 
                                          max_items=40)
search_result2

[<Item title:"bigDataFileShares_ServiceCallsOrleans" type:Big Data File Share owner:admin>]

In [5]:
data_item = search_result2[0]
data_item

<Item title:"bigDataFileShares_ServiceCallsOrleans" type:Big Data File Share owner:admin>

In [6]:
#displays layers in the item
data_item.layers

[<Layer url:"https://ndhwks6.esri.com/server/rest/services/DataStoreCatalogs/bigDataFileShares_ServiceCallsOrleans/BigDataCatalogServer/calls">]

In [7]:
calls = data_item.layers[0] #select first layer 
calls

<Layer url:"https://ndhwks6.esri.com/server/rest/services/DataStoreCatalogs/bigDataFileShares_ServiceCallsOrleans/BigDataCatalogServer/calls">

In [8]:
calls.properties

{
  "dataStoreID": "c7ccc770-5128-4c2f-a4ae-cee8f057d52f",
  "fields": [
    {
      "name": "NOPD_Item",
      "type": "esriFieldTypeString"
    },
    {
      "name": "Type_",
      "type": "esriFieldTypeString"
    },
    {
      "name": "TypeText",
      "type": "esriFieldTypeString"
    },
    {
      "name": "Priority",
      "type": "esriFieldTypeString"
    },
    {
      "name": "MapX",
      "type": "esriFieldTypeDouble"
    },
    {
      "name": "MapY",
      "type": "esriFieldTypeDouble"
    },
    {
      "name": "TimeCreate",
      "type": "esriFieldTypeString"
    },
    {
      "name": "TimeDispatch",
      "type": "esriFieldTypeString"
    },
    {
      "name": "TimeArrive",
      "type": "esriFieldTypeString"
    },
    {
      "name": "TimeClosed",
      "type": "esriFieldTypeString"
    },
    {
      "name": "Disposition",
      "type": "esriFieldTypeString"
    },
    {
      "name": "DispositionText",
      "type": "esriFieldTypeString"
    },
    {
      "name

## Calculate Density

In [9]:
from arcgis.geoanalytics.analyze_patterns import calculate_density
from datetime import datetime as dt

In [10]:
##usage example
density = calculate_density(calls, 
                            bin_size=1,
                            bin_size_unit='Miles',
                            radius=100,
                            radius_unit='Miles',
                            output_name='calculate density' + str(dt.now().microsecond))
density

<Item title:"calculate_density970774" type:Feature Layer Collection owner:admin>

In [11]:
density.delete()

True

#### Create Space Time Cube

In [12]:
from arcgis.geoanalytics.analyze_patterns import create_space_time_cube

In [13]:
##usage example
create_space_time_cube(point_layer=calls,
                       bin_size=100,
                       bin_size_unit="Miles",
                       time_step_interval=1,
                       time_step_interval_unit="Days",
                       time_step_alignment='StartTime',
                       output_name="space_time_cube")

{"url": "https://ndhwks6.esri.com/server/rest/directories/arcgisjobs/system/geoanalyticstools_gpserver/j3a92886c40094860b039cbcc0d778d4c/scratch/space_time_cube.nc"}

#### Find Hot Spots

In [14]:
from arcgis.geoanalytics.analyze_patterns import find_hot_spots
from datetime import datetime as dt

In [15]:
##usage example
hot_spots = find_hot_spots(calls, 
                           bin_size=100,
                           bin_size_unit='Meters',
                           neighborhood_distance=250,
                           neighborhood_distance_unit='Meters',
                           output_name="get hot spot areas" + str(dt.now().microsecond))
hot_spots

<Item title:"get_hot_spot_areas4416" type:Feature Layer Collection owner:admin>

In [17]:
hot_spots.delete()

True

#### Find Point Clusters

In [18]:
from arcgis.geoanalytics.analyze_patterns import find_point_clusters

In [19]:
point_clusters = find_point_clusters(calls, 
                                     method='HDBSCAN', 
                                     min_feature_clusters=5, 
                                     output_name='point clusters' + str(dt.now().microsecond))

In [23]:
point_clusters.delete()

True

#### Forest

In [20]:
from arcgis.geoanalytics.analyze_patterns import forest

In [21]:
##usage example
forest = forest(input_layer=data_layer,
       var_prediction={"fieldName":"capacity_f", "categorical":False},
       var_explanatory=[{"fieldName":"altitude_m", "categorical":False},
                        {"fieldName":"wind_speed", "categorical":False},
                        {"fieldName":"dayl__s_", "categorical":False},
                        {"fieldName":"prcp__mm_d", "categorical":False},
                        {"fieldName":"srad__W_m_", "categorical":False},
                        {"fieldName":"swe__kg_m_", "categorical":False},
                        {"fieldName":"tmax__deg", "categorical":False},
                        {"fieldName":"tmin__deg", "categorical":False},
                        {"fieldName":"vp__Pa_", "categorical":False}
                       ],
       prediction_type="TrainAndPredict",
       trees=200,
       importance_tbl=True)

<Item title:"Forest_Based_Regression_3SZP85" type:Feature Layer Collection owner:admin>

In [22]:
forest.delete()

True

#### GLR

In [24]:
from arcgis.geoanalytics.analyze_patterns import glr

In [25]:
glr_output = glr(data_layer, 
                 var_dependent='capacity_f',
                 var_explanatory=['altitude_m', 'wind_speed', 'dayl__s_', 'prcp__mm_d',
                                  'srad__W_m_','swe__kg_m_','tmax__deg','tmin__deg','vp__Pa_'],
                 output_name='glr' + str(dt.now().microsecond))
glr_output

<Item title:"glr859111" type:Feature Layer Collection owner:admin>

In [26]:
glr_output.delete()

True

#### GWR

In [27]:
from arcgis.geoanalytics.analyze_patterns import gwr

In [28]:
bigdata_datastore_manager = arcgis.geoanalytics.get_datastores()
bigdata_datastore_manager

<DatastoreManager for https://ndhwks6.esri.com:6443/arcgis/admin>

In [29]:
data_item2 = bigdata_datastore_manager.add_bigdata("all_hurricanes", r"\\NDHWKS6\Users\arcgis\Documents\hurricanes_1848_2010")

Big Data file share exists for all_hurricanes


In [30]:
search_result = portal_gis.content.search("bigDataFileShares_all_hurricanes", 
                                          item_type = "big data file share", 
                                          max_items=40)
data_item = search_result[0]
data_item

<Item title:"bigDataFileShares_all_hurricanes" type:Big Data File Share owner:admin>

In [31]:
hurricanes = data_item.layers[0] #select first layer 
hurricanes

<Layer url:"https://ndhwks6.esri.com/server/rest/services/DataStoreCatalogs/bigDataFileShares_all_hurricanes/BigDataCatalogServer/hurricanes">

In [32]:
hurricanes.properties

{
  "dataStoreID": "2d305bf2-3bd4-4e33-ac95-7bb5d987b494",
  "fields": [
    {
      "name": "serial_num",
      "type": "esriFieldTypeString"
    },
    {
      "name": "season",
      "type": "esriFieldTypeInteger"
    },
    {
      "name": "num",
      "type": "esriFieldTypeInteger"
    },
    {
      "name": "basin",
      "type": "esriFieldTypeString"
    },
    {
      "name": "sub_basin",
      "type": "esriFieldTypeString"
    },
    {
      "name": "name",
      "type": "esriFieldTypeString"
    },
    {
      "name": "iso_time",
      "type": "esriFieldTypeString"
    },
    {
      "name": "nature",
      "type": "esriFieldTypeString"
    },
    {
      "name": "latitude",
      "type": "esriFieldTypeDouble"
    },
    {
      "name": "longitude",
      "type": "esriFieldTypeDouble"
    },
    {
      "name": "wind_wmo_",
      "type": "esriFieldTypeDouble"
    },
    {
      "name": "pres_wmo_",
      "type": "esriFieldTypeInteger"
    },
    {
      "name": "center",
    

In [33]:
arcgis.env.process_spatial_reference=54034

In [34]:
#get new data
gwr_output = gwr(hurricanes, 
                 dependent_variable=['season'], 
                 explanatory_variables=['wind', 'wind_wmo1', 'pres_wmo_'],
                 number_of_neighbors=100,
                 output_name='gwr' + str(dt.now().microsecond))
gwr_output

<Item title:"gwr125078" type:Feature Layer Collection owner:admin>

In [35]:
gwr_output.delete()

True